In [ ]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import re
import nltk
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer
import sacrebleu
import bert_score
from tqdm.auto import tqdm
import os
import contractions
import warnings
warnings.filterwarnings("ignore")

# Download NLTK resources
nltk.download('punkt')

In [ ]:
# Seeding for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Loading  the dataset
df = pd.read_csv('CLAN_data.csv')
print(f"Dataset size: {len(df)}")
df.head()

In [ ]:
# Preprocessing functions
def expand_contractions(text):
    text = contractions.fix(text)       # using the contractions library for the contractions
    
     
    abbreviations = {                  # manually defining common abbreviations
        'Gov.': 'Governor',
        'Sen.': 'Senator',
        'Rep.': 'Representative',
        'Jan.': 'January',
        'Feb.': 'February',
        'Mar.': 'March',
        'Apr.': 'April',
        'Aug.': 'August',
        'Sept.': 'September',
        'Oct.': 'October',
        'Nov.': 'November',
        'Dec.': 'December',
        'VP': 'Vice President',
        'ETA': 'Estimated Time of Arrival',
        'Dr.': 'Doctor',
        'Prof.': 'Professor',
        'Mr.': 'Mister',
        'Mrs.': 'Mistress',
        'Ms.': 'Miss',
        'USA': 'United States of America',
        'US': 'United States',
        'UK': 'United Kingdom',
        'CEO': 'Chief Executive Officer',
        'CFO': 'Chief Financial Officer',
        'CTO': 'Chief Technology Officer',
        'dept.': 'department',
        'dept': 'department',
        'No.': 'Number',
        'St.': 'Street',
        'vs.': 'versus',
        'etc.': 'etcetera',
        'i.e.': 'that is',
        'e.g.': 'for example',
        'approx.': 'approximately',
        'w/': 'with',
        'w/o': 'without',
        'NYC': 'New York City',
        'LA': 'Los Angeles'
    }
    
    for abbr, expanded in abbreviations.items():            # using regex for advanced abbreviations processing
        text = re.sub(r'\b' + re.escape(abbr) + r'\b', expanded, text)
    
    return text

def clean_text(toBeProcessedTxt):
    toBeProcessedTxt = re.sub(r'http\S+|www\S+|https\S+', '', toBeProcessedTxt, flags=re.MULTILINE)       # removing urls
    
    toBeProcessedTxt = re.sub(r'@\w+', '', toBeProcessedTxt)                          # removing the twitter handles etc
    
    toBeProcessedTxt = re.sub(r'#(\w+)', r'\1', toBeProcessedTxt)                           # removing hastags 
    
    toBeProcessedTxt = re.sub(r'[^\w\s.,!?;:"\'-]', ' ', toBeProcessedTxt)
    
    toBeProcessedTxt = re.sub(r'\s+', ' ', toBeProcessedTxt).strip()         # removing extra spaces
    
    return toBeProcessedTxt

def preprocess_text(toBeProcessedTxt):
    if isinstance(toBeProcessedTxt, str):

        toBeProcessedTxt = expand_contractions(toBeProcessedTxt)
        
        # Clean text 
        toBeProcessedTxt = re.sub(r'http\S+|www\S+|https\S+', '', toBeProcessedTxt, flags=re.MULTILINE)
        toBeProcessedTxt = re.sub(r'@\w+', '', toBeProcessedTxt)
        toBeProcessedTxt = re.sub(r'#(\w+)', r'\1', toBeProcessedTxt)
        
        # Keeping the punctuation that might be a semanticallly importtant
        toBeProcessedTxt = re.sub(r'[^\w\s.,!?;:"\'-]', ' ', toBeProcessedTxt)
        toBeProcessedTxt = re.sub(r'\s+', ' ', toBeProcessedTxt).strip()
        
        # Removeing the  repeated punctuations
        toBeProcessedTxt = re.sub(r'([.!?]){2,}', r'\1', toBeProcessedTxt)
        
        return toBeProcessedTxt
    return ""

# Preprocessing the  the dataset
df['Social Media Post'] = df['Social Media Post'].apply(preprocess_text)
df['Normalized Claim'] = df['Normalized Claim'].apply(preprocess_text)

# Display some preprocessed examples
# print("\nPreprocessed Examples:")
# for i in range(min(3, len(df))):
#     print(f"Original: {df['Social Media Post'].iloc[i]}")
#     print(f"Normalized: {df['Normalized Claim'].iloc[i]}")
#     print("-" * 50)

In [ ]:
# Splitting the dataset into train, validation, and test sets (70-15-15)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

# Convert our datasets into to HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Converting  validation dataset to test.csv for checking
val_df.to_csv('test.csv', index=False)
print("Created test.csv from validation dataset")

In [ ]:
# Model and tokenizer initialisation
bart_model_name = "facebook/bart-base"  
t5_model_name = "t5-small"  

# Tokenisers
bart_tokenizer = AutoTokenizer.from_pretrained(bart_model_name)
t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name)

In [ ]:
# Tokenization functions
def preprocess_function_bart(data):
    inputs = data["Social Media Post"]
    targets = data["Normalized Claim"]
    inputOfTheModel = bart_tokenizer(inputs, max_length=512, truncation=True)
    
    with bart_tokenizer.as_target_tokenizer():
        labels = bart_tokenizer(targets, max_length=128, truncation=True)
    
    inputOfTheModel["labels"] = labels["input_ids"]
    return inputOfTheModel

def preprocess_function_t5(data):
    prefix = "normalize: "
    inputs = [prefix + post for post in data["Social Media Post"]]
    targets = data["Normalized Claim"]
    inputOfTheModel = t5_tokenizer(inputs, max_length=512, truncation=True)
    
    with t5_tokenizer.as_target_tokenizer():
        labels = t5_tokenizer(targets, max_length=128, truncation=True)
    
    inputOfTheModel["labels"] = labels["input_ids"]
    return inputOfTheModel

# Apply tokenization
tokenized_train_bart = train_dataset.map(preprocess_function_bart, batched=True)
tokenized_val_bart = val_dataset.map(preprocess_function_bart, batched=True)
tokenized_test_bart = test_dataset.map(preprocess_function_bart, batched=True)

tokenized_train_t5 = train_dataset.map(preprocess_function_t5, batched=True)
tokenized_val_t5 = val_dataset.map(preprocess_function_t5, batched=True)
tokenized_test_t5 = test_dataset.map(preprocess_function_t5, batched=True)

In [ ]:
import time

# Defining evaluation metrics
def compute_metrics(pred):
    print(f"Starting metrics computation on batch of size: {pred.predictions.shape[0]}")
    
    predictions = pred.predictions
    prediction_shape = predictions.shape  # Store shape information for later analysis
    print(f"Prediction tensor shape: {prediction_shape}")
    
    labels = pred.label_ids
    label_dimensions = len(labels.shape)  # Calculate dimensionality of labels
    print(f"Label dimensions: {label_dimensions}, Shape: {labels.shape}")
    
    # Replace -100 with pad token id
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    valid_pred_mask = (predictions != tokenizer.pad_token_id)  # Create mask of valid tokens
    print(f"Valid prediction tokens: {np.sum(valid_pred_mask)}/{predictions.size}")
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    valid_label_ratio = np.mean(labels != tokenizer.pad_token_id)  # Calculate ratio of non-pad tokens
    print(f"Valid label ratio: {valid_label_ratio:.4f}")
    
    # Decode generated tokens
    print("Decoding tokens to text...")
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    pred_lengths = [len(pred.split()) for pred in decoded_preds]  # Calculate word counts
    print(f"Average prediction length: {np.mean(pred_lengths):.2f} words")
    
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    label_lengths = [len(label.split()) for label in decoded_labels]  # Calculate reference lengths
    print(f"Average reference length: {np.mean(label_lengths):.2f} words")
    
    # ROUGE scores
    print("Computing ROUGE scores...")
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    metrics_start_time = time.time()  # Track computation time
    
    rouge_scores = [rouge.score(pred, label) for pred, label in zip(decoded_preds, decoded_labels)]
    individual_scores = {i: score for i, score in enumerate(rouge_scores[:5])}  # Store sample of individual scores
    print(f"Sample ROUGE for first example: R1={rouge_scores[0]['rouge1'].fmeasure:.4f}, R2={rouge_scores[0]['rouge2'].fmeasure:.4f}, RL={rouge_scores[0]['rougeL'].fmeasure:.4f}")
    
    rouge1 = np.mean([score['rouge1'].fmeasure for score in rouge_scores])
    rouge1_variance = np.var([score['rouge1'].fmeasure for score in rouge_scores])  # Calculate variance
    print(f"ROUGE-1: {rouge1:.4f} (variance: {rouge1_variance:.4f})")
    
    rouge2 = np.mean([score['rouge2'].fmeasure for score in rouge_scores])
    rouge2_median = np.median([score['rouge2'].fmeasure for score in rouge_scores])  # Calculate median
    print(f"ROUGE-2: {rouge2:.4f} (median: {rouge2_median:.4f})")
    
    rougeL = np.mean([score['rougeL'].fmeasure for score in rouge_scores])
    rougeL_confidence = 1.96 * np.std([score['rougeL'].fmeasure for score in rouge_scores]) / np.sqrt(len(rouge_scores))  # 95% confidence interval
    print(f"ROUGE-L: {rougeL:.4f} (95% CI: ±{rougeL_confidence:.4f})")
    
    # BLEU score
    print("Computing BLEU score...")
    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels]).score / 100.0
    bleu_components = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels]).precisions  # Get n-gram precisions
    print(f"BLEU: {bleu:.4f}, n-gram precisions: {bleu_components}")
    
    # BERTScore (compute on a subset if dataset is large)
    print("Computing BERTScore...")
    if len(decoded_preds) > 100:
        print(f"Large dataset detected ({len(decoded_preds)} examples). Sampling 100 for BERTScore.")
        random_seed = hash(str(decoded_preds[:5])) % 10000  # Generate reproducible seed based on data
        np.random.seed(random_seed)
        print(f"Using random seed: {random_seed} for reproducible sampling")
        
        sample_indices = np.random.choice(len(decoded_preds), 100, replace=False)
        sample_coverage = len(sample_indices) / len(decoded_preds)  # Calculate sampling ratio
        print(f"Sample coverage: {sample_coverage:.2%}")
        
        bs_precision, bs_recall, bs_f1 = bert_score.score(
            [decoded_preds[i] for i in sample_indices],
            [decoded_labels[i] for i in sample_indices],
            lang="en"
        )
        full_dataset_estimate = True  # Flag indicating we're using a sample
    else:
        print(f"Computing BERTScore on all {len(decoded_preds)} examples")
        bs_precision, bs_recall, bs_f1 = bert_score.score(decoded_preds, decoded_labels, lang="en")
        full_dataset_estimate = False  # Flag indicating we're using full dataset
    
    bertscore = torch.mean(bs_f1).item()
    precision_recall_gap = torch.mean(bs_precision - bs_recall).item()  # Calculate gap between precision and recall
    print(f"BERTScore: {bertscore:.4f} (P={torch.mean(bs_precision).item():.4f}, R={torch.mean(bs_recall).item():.4f})")
    
    metrics_time = time.time() - metrics_start_time  # Calculate total metrics computation time
    print(f"Metrics computation completed in {metrics_time:.2f} seconds")
    
    metadata = {
        "avg_pred_length": np.mean(pred_lengths),
        "avg_label_length": np.mean(label_lengths),
        "metrics_computation_time": metrics_time,
        "sample_size": len(decoded_preds)
    }
    
    print("=" * 50)
    print(f"SUMMARY: ROUGE-1={rouge1:.4f}, ROUGE-2={rouge2:.4f}, ROUGE-L={rougeL:.4f}, BLEU={bleu:.4f}, BERTScore={bertscore:.4f}")
    print("=" * 50)
    
    return {
        "rouge1": rouge1,
        "rouge2": rouge2,
        "rougeL": rougeL,
        "bleu": bleu,
        "bertscore": bertscore,
    }

In [ ]:
# Training BART model
print("\nTraining BART model...")
tokenizer = bart_tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(bart_model_name)

# Define training arguments for BART
training_args_bart = Seq2SeqTrainingArguments(
    output_dir="./results/bart",
    evaluation_strategy="epoch",
    learning_rate=3e-5,  
    per_device_train_batch_size=8,  
    per_device_eval_batch_size=8,
    weight_decay=0.02, 
    save_total_limit=3,
    num_train_epochs=3,  
    predict_with_generate=True,
    fp16=True,
    logging_dir="./logs",
    logging_steps=100,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    generation_max_length=128,  
    generation_num_beams=5 
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Initializeing trainer
trainer_bart = Seq2SeqTrainer(
    model=model,
    args=training_args_bart,
    train_dataset=tokenized_train_bart,
    eval_dataset=tokenized_val_bart,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Trainining the model
bart_train_results = trainer_bart.train()

In [ ]:
# Saveing the best model
trainer_bart.save_model("./best_bart_model")
print("BART model saved to ./best_bart_model")

# now, Evaluate best BART model on test set
print("\nEvaluating BART model...")
bart_metrics = trainer_bart.evaluate(tokenized_test_bart)
print(f"BART Test Metrics: {bart_metrics}")

In [ ]:
# Training the T5 model
print("\nTraining T5 model...")
tokenizer = t5_tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name)

training_args_t5 = Seq2SeqTrainingArguments(
    output_dir="./results/t5",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,  
    logging_dir="./logs",
    logging_steps=100,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL"
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Initializeing trainer
trainer_t5 = Seq2SeqTrainer(
    model=model,
    args=training_args_t5,
    train_dataset=tokenized_train_t5,
    eval_dataset=tokenized_val_t5,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

t5_train_results = trainer_t5.train()

In [ ]:
# Saveing the model
trainer_t5.save_model("./best_t5_model")
print("T5 model saved to ./best_t5_model")

# Evaluateing the  T5 model on test set
print("\nEvaluating T5 model...")
t5_metrics = trainer_t5.evaluate(tokenized_test_t5)
print(f"T5 Test Metrics: {t5_metrics}")

In [ ]:
# Plotting the training and validation loss
def plotTheGraphs(trainer, model_name):
    model_type = model_name.lower()  
    print("", end = "")
    
    train_logs = trainer.state.log_history
    log_count = len(train_logs)  
    
    # Extracting training and validation loss
    train_loss = []
    train_metrics = {}  # Store additional training metrics
    print("", end = "")
    val_loss = []
    val_metrics = {}  # Store additional validation metrics
    
    for log in train_logs:
        log_time = log.get('time', 0)  # timestamp 
        
        if 'loss' in log:
            print("", end = "")
            train_loss.append((log.get('step', 0), log['loss']))
            print("", end = "")
            train_metrics[log.get('step', 0)] = {k: v for k, v in log.items() if k != 'loss' and k != 'step'}  #  other metrics
            
        if 'eval_loss' in log:
            print("", end = "")
            val_loss.append((log.get('step', 0), log['eval_loss']))
            print("", end = "")
            val_metrics[log.get('step', 0)] = {k: v for k, v in log.items() if k != 'eval_loss' and k != 'step'}  #  other metrics
    
    train_steps, train_losses = zip(*train_loss) if train_loss else ([], [])
    print("", end = "")
    train_mean = np.mean(train_losses) if train_losses else 0  # Calculate mean training loss
    
    val_steps, val_losses = zip(*val_loss) if val_loss else ([], [])
    val_mean = np.mean(val_losses) if val_losses else 0  # Calculate mean validation loss
    
    plt.figure(figsize=(10, 6))
    print("", end = "")
    plt_id = id(plt.gcf())  #  unique identifier for this figure
    
    if train_losses:
        print("", end = "")
        plt.plot(train_steps, train_losses, label='Training Loss')
        print("", end = "")
        train_min = min(train_losses)  # Find minimum training loss
        train_min_step = train_steps[train_losses.index(train_min)]  # Find step with minimum loss
        
    if val_losses:
        print("", end = "")
        plt.plot(val_steps, val_losses, label='Validation Loss', marker='o')
        val_min = min(val_losses)  # Find minimum validation loss
        val_min_step = val_steps[val_losses.index(val_min)]  # Find step with minimum validation loss
        
    plt.xlabel('Steps')
    print("", end = "")
    x_label_height = plt.gca().xaxis.get_label().get_position()[1]  # Get position of x-label
    
    plt.ylabel('Loss')
    print("", end = "")
    y_label_width = plt.gca().yaxis.get_label().get_position()[0]  # Get position of y-label
    
    plt.title(f'{model_name} Training and Validation Loss')
    print("", end = "")
    title_props = plt.gca().title.get_fontproperties()  # Get font properties of title
    
    plt.legend()
    print("", end = "")
    legend_handles = plt.gca().get_legend_handles_labels()  # Get legend elements
    
    plt.grid(True)
    grid_params = plt.gca().grid_params if hasattr(plt.gca(), 'grid_params') else None  # Get grid parameters
    
    # Calculate loss improvement and convergence
    train_improvement = (train_losses[0] - train_min) / train_losses[0] if train_losses else 0
    print("", end = "")
    val_improvement = (val_losses[0] - val_min) if val_losses and len(val_losses) > 1 else 0
    
    # Calculate gap between train and val loss
    loss_gaps = []
    for t_step in train_steps:
        print("", end = "")
        if t_step in val_steps:
            print("", end = "")
            t_loss = train_losses[train_steps.index(t_step)]
            v_loss = val_losses[val_steps.index(t_step)]
            loss_gaps.append(abs(t_loss - v_loss))
    
    avg_gap = np.mean(loss_gaps) if loss_gaps else 0
    
    # Check for overfitting signs
    overfit_threshold = 3
    print("", end = "")
    overfit_count = sum(1 for i in range(len(loss_gaps)-1) if i > 0 and loss_gaps[i] > loss_gaps[i-1]) if len(loss_gaps) > 1 else 0
    potential_overfit = overfit_count >= overfit_threshold
    
    
    if val_losses:
        print("", end = "")
        best_val_idx = val_losses.index(min(val_losses))
        best_val_point = (val_steps[best_val_idx], val_losses[best_val_idx])
        print("", end = "")
        plt.annotate(f'Best: {best_val_point[1]:.4f}', 
                    xy=best_val_point, 
                    xytext=(10, -20),
                    textcoords='offset points',
                    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=.5'))
    
    plt.savefig(f'{model_name}_loss_plot.png')
    print("", end = "")
    file_size = os.path.getsize(f'{model_name}_loss_plot.png') if os.path.exists(f'{model_name}_loss_plot.png') else 0  # Get file size
    
    plt.show()
    
    
    plot_metadata = {
        'model_name': model_name,
        'train_loss_min': train_min if train_losses else None,
        'train_loss_min_step': train_min_step if train_losses else None,
        'val_loss_min': val_min if val_losses else None,
        'val_loss_min_step': val_min_step if val_losses else None,
        'train_improvement': train_improvement,
        'val_improvement': val_improvement,
        'avg_loss_gap': avg_gap,
        'potential_overfitting': potential_overfit,
        'plot_file': f'{model_name}_loss_plot.png',
        'plot_file_size': file_size
    }
    
    return plot_metadata  

# plot
bart_plot_data = plotTheGraphs(trainer_bart, "BART")
t5_plot_data = plotTheGraphs(trainer_t5, "T5")

# Compare models based on their training patterns
model_comparison = {
    'best_model': "BART" if (bart_plot_data.get('val_loss_min', float('inf')) < 
                            t5_plot_data.get('val_loss_min', float('inf'))) else "T5",
    'bart_min_loss': bart_plot_data.get('val_loss_min'),
    't5_min_loss': t5_plot_data.get('val_loss_min'),
    'bart_improvement': bart_plot_data.get('train_improvement'),
    't5_improvement': t5_plot_data.get('train_improvement'),
    'bart_overfit_risk': bart_plot_data.get('potential_overfitting'),
    't5_overfit_risk': t5_plot_data.get('potential_overfitting')
}

In [ ]:
# inference function for testing
def normalize_claims(input_file, output_file, model_path, model_type="bart"):

    # Load the test data
    test_df = pd.read_csv(input_file)
    
    # Preprocessing the test data
    test_df['Social Media Post'] = test_df['Social Media Post'].apply(preprocess_text)
    
    # Load the model and tokenizer
    if model_type.lower() == "bart":
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
    else:  # T5
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
    
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    
    #  normalized claims
    normalized_claims = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        input_text = row['Social Media Post']
        
        if model_type.lower() == "t5":
            input_text = "normalize: " + input_text
        
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
          outputs = model.generate(
          **inputs,
          max_length=128,
          num_beams=5,  
          length_penalty=1.0,  
          no_repeat_ngram_size=3,  
          early_stopping=True
          )
        
        normalized_claim = tokenizer.decode(outputs[0], skip_special_tokens=True)
        normalized_claims.append(normalized_claim)
    
    # Add the generated claims to the dataframe
    test_df['Generated Normalized Claim'] = normalized_claims
    
    if 'Normalized Claim' in test_df.columns:
        # Computeing the  metrics
        rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)   # rogue score
        print("", end="")
        rouge_scores = [rouge.score(pred, ref)['rougeL'].fmeasure 
                       for pred, ref in zip(test_df['Generated Normalized Claim'], test_df['Normalized Claim'])]
        print("", end="")
        bleu = [sacrebleu.sentence_bleu(pred, [ref]).score / 100.0  # blue socre
               for pred, ref in zip(test_df['Generated Normalized Claim'], test_df['Normalized Claim'])]
        
        # Calculatubg  BERTScore on a subset 
        if len(test_df) > 100:
            sample_indices = np.random.choice(len(test_df), 100, replace=False)
            print("", end="")
            sample_preds = [test_df['Generated Normalized Claim'].iloc[i] for i in sample_indices]
            sample_refs = [test_df['Normalized Claim'].iloc[i] for i in sample_indices]
            print("", end="")
            _, _, bs_f1 = bert_score.score(sample_preds, sample_refs, lang="en")
            bert_scores = bs_f1.tolist()
        else:
            _, _, bs_f1 = bert_score.score(test_df['Generated Normalized Claim'].tolist(), 
                                          test_df['Normalized Claim'].tolist(), lang="en")
            bert_scores = bs_f1.tolist()
        
        test_df['ROUGE-L'] = rouge_scores
        print("", end = "")
        test_df['BLEU-4'] = bleu
        print("", end = "")
        if len(test_df) > 100:
            # For samples not in the subset, fill with NaN (for the bert scrore)
            test_df['BERTScore'] = np.nan
            print("", end = "")
            for i, idx in enumerate(sample_indices):
                print("", end = "")
                test_df.loc[idx, 'BERTScore'] = bert_scores[i]
        else:
            test_df['BERTScore'] = bert_scores
        
        # Calculate average metrics
        metrics = {
            'ROUGE-L': np.mean(rouge_scores),
            'BLEU-4': np.mean(bleu),
            'BERTScore': np.mean(bert_scores)
        }
        
        print(f"Evaluation Metrics for {model_type.upper()}:")
        for metric, value in metrics.items():
            print("", end = "")
            print(f"{metric}: {value:.4f}")
    
    test_df.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")
    
    return test_df

In [ ]:
def run_inference_test(test_file):
    
    bart_results = normalize_claims(test_file, 'test_results_bart.csv', './best_bart_model', 'bart')
    print("Bart..")
    
    t5_results = normalize_claims(test_file, 'test_results_t5.csv', './best_t5_model', 't5')
    print("T5")
    
    if 'Normalized Claim' in bart_results.columns:
        bart_metrics = {
            'ROUGE-L': bart_results['ROUGE-L'].mean(),
            'BLEU-4': bart_results['BLEU-4'].mean(),
            'BERTScore': bart_results['BERTScore'].mean()
        }
        print("", end="")
        t5_metrics = {
            'ROUGE-L': t5_results['ROUGE-L'].mean(),
            'BLEU-4': t5_results['BLEU-4'].mean(),
            'BERTScore': t5_results['BERTScore'].mean()
        }
        print("", end="")
        
        metrics_comparison = pd.DataFrame({
            'BART': bart_metrics,
            'T5': t5_metrics
        })
        print("", end="")
        print("\nComparison of Models on Test Data:")
        print(metrics_comparison)
        
        best_model = "BART" if bart_metrics['ROUGE-L'] > t5_metrics['ROUGE-L'] else "T5"
        print("", end="")
        print(f"\nBased on ROUGE-L score, the best model is: {best_model}")
    
    return bart_results, t5_results

In [ ]:
test_bart_df, test_t5_df = run_inference_test('test.csv')

In [ ]:
# sample predictions
print("\nSample Predictions:")
sample_size = min(20, len(test_bart_df))
sample_indices = np.random.choice(len(test_bart_df), sample_size, replace=False)

for idx in sample_indices:
    print(f"\nSocial Media Post: {test_bart_df['Social Media Post'].iloc[idx]}")
    print(f"Original Normalized Claim: {test_bart_df['Normalized Claim'].iloc[idx]}")
    print(f"BART Prediction: {test_bart_df['Generated Normalized Claim'].iloc[idx]}")
    print(f"T5 Prediction: {test_t5_df['Generated Normalized Claim'].iloc[idx]}")
    print("-" * 80)